# Visualizar a base — projeto energia 2025
Execute as células de cima para baixo (*Ambiente de execução → Executar tudo*).

In [ ]:
# 1) Drive + extração do zip (procura na pasta do grupo, em Drives compartilhados, atalhos e Meu Drive)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # relê o Drive (pega uploads recentes)
import os, glob, zipfile, shutil

TAM_OK = 88051547            # tamanho exato do zip correto (versão de 2026-10-09)
PASTA_GRUPO = 'ENG4040_33A_GRUPOB'
ZIP_MANUAL = None            # se a busca falhar, cole aqui o caminho do zip, ex.: '/content/drive/Shareddrives/ENG4040_33A_GRUPOB/sprint 2/projeto_energia_csv_2025.zip'

def acha_zip():
    if ZIP_MANUAL:
        return ZIP_MANUAL
    raizes = ['/content/drive/Shareddrives', '/content/drive/MyDrive', '/content/drive/.shortcut-targets-by-id']
    vistos = []
    # 1º: dentro da pasta do grupo (drive compartilhado, atalho ou Meu Drive); 2º: qualquer lugar
    for padrao in (f'**/{PASTA_GRUPO}/**/*.zip', '**/projeto_energia_csv_2025*.zip'):
        for r in raizes:
            if not os.path.isdir(r):
                continue
            for p in glob.glob(f'{r}/{padrao}', recursive=True):
                t = os.path.getsize(p); vistos.append((t, p))
                if t == TAM_OK:
                    return p
    print('zips encontrados (nenhum com o tamanho certo):'); [print(f'  {t:>12,d}  {p}') for t, p in vistos]
    raise FileNotFoundError(
        f'Não achei o zip de {TAM_OK} bytes. Se a pasta {PASTA_GRUPO} foi "compartilhada comigo", '
        'clique nela com o botão direito no Drive → Organizar → Adicionar atalho ao Meu Drive, e rode de novo. '
        'Ou preencha ZIP_MANUAL com o caminho do zip.')

Z = acha_zip()
assert os.path.getsize(Z) == TAM_OK, f'{Z} não é a versão correta ({os.path.getsize(Z)} bytes)'
D = '/content/extracted_data'
shutil.rmtree(D, ignore_errors=True)
zipfile.ZipFile(Z).extractall(D)
print('usando:', Z, '|', len(os.listdir(D)), 'tabelas')

In [ ]:
# 2) Visão geral: todas as tabelas, com linhas, colunas e tamanho
import pandas as pd
from google.colab import data_table
data_table.enable_dataframe_formatter()   # tabelas viram planilhas interativas (filtrar/ordenar)
resumo = []
for f in sorted(os.listdir(D)):
    df = pd.read_csv(f'{D}/{f}', nrows=5)
    n = sum(1 for _ in open(f'{D}/{f}')) - 1
    resumo.append(dict(tabela=f.replace('.csv', ''), linhas=n, colunas=df.shape[1],
                       MB=round(os.path.getsize(f'{D}/{f}') / 1e6, 1), algumas_colunas=', '.join(df.columns[:6])))
pd.DataFrame(resumo)

In [ ]:
# 3) Painel por subsistema — troque SUB e MES para explorar
p = pd.read_csv(f'{D}/painel_subsistema_h.csv', parse_dates=['ts_local'])
SUB, MES = 'NE', 8
p[(p['subsistema'] == SUB) & (p['ts_local'].dt.month == MES)]

In [ ]:
# 4) Usinas (dim_usina) — filtre/ordene na tabela interativa
pd.read_csv(f'{D}/dim_usina.csv')

In [ ]:
# 5) Gráficos rápidos
import matplotlib.pyplot as plt
p['mes'] = p['ts_local'].dt.month; p['hora'] = p['ts_local'].dt.hour
fontes = ['ger_hidro_mwh', 'ger_eolica_mwh', 'ger_solar_mwh', 'ger_fossil_mwh', 'ger_biomassa_mwh', 'ger_nuclear_mwh']
fig, ax = plt.subplots(1, 3, figsize=(20, 5))
(p.groupby('mes')[fontes].sum() / 1e6).plot.bar(stacked=True, ax=ax[0], title='Geração do SIN por mês (TWh)')
ne = p[p['subsistema'] == 'NE'].groupby('hora')
ne[['ger_eolica_mwh', 'ger_solar_mwh', 'ger_hidro_mwh']].mean().plot.area(ax=ax[1], title='NE — dia médio (MW)')
ne['carga_global_mwh'].mean().plot(ax=ax[1], color='k', lw=2, label='carga'); ax[1].legend()
q = p[p['subsistema'] == 'NE']
cor = q.pivot_table(index='mes', columns='hora', values='curtail_corte_mwh_eolica', aggfunc='sum') / q.pivot_table(index='mes', columns='hora', values='ref_eolica_mwh', aggfunc='sum')
im = ax[2].imshow(cor, aspect='auto', origin='lower', extent=[-0.5, 23.5, 0.5, 12.5]); fig.colorbar(im, ax=ax[2])
ax[2].set_title('NE — fração da eólica cortada (mês × hora)'); ax[2].set_xlabel('hora'); ax[2].set_ylabel('mês')
plt.tight_layout(); plt.show()